# Авторское решение итогового проекта

Решение показывает полный путь четырёхзанятийного проекта: постановку, предобработку без утечки, сравнение моделей, подбор параметров, анализ ошибок, финальный сабмит и материал для защиты.

Это не единственно правильный путь. Его задача — дать преподавателю воспроизводимый ориентир и набор идей для разбора после завершения соревнования.


## 1. Задача и данные

Нужно оценить вероятность ухода пользователя образовательного сервиса в ближайший месяц. Это бинарная классификация; цель — `churn`, а официальная метрика StackMoreLayers — ROC-AUC.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

target = "churn"
id_col = "id"

print("train:", train.shape, "test:", test.shape)
print("Доля класса 1:", round(train[target].mean(), 3))
display(train.head())


## 2. Признаки и честное разбиение

Все новые признаки строятся одинаковой функцией для train и test и не используют цель. Validation отделяется до обучения кодировщиков и моделей.


In [ ]:
def add_features(df):
    df = df.copy()
    df["sessions_change"] = df["sessions_last_30"] - df["sessions_previous_30"]
    df["sessions_retention"] = (df["sessions_last_30"] + 1) / (df["sessions_previous_30"] + 1)
    df["activity_drop"] = (
        df["sessions_last_30"] <= np.maximum(2, 0.5 * df["sessions_previous_30"])
    ).astype(int)
    df["sessions_per_100_days"] = df["sessions_last_30"] / (df["account_age_days"] / 100 + 1)
    df["price_after_discount"] = df["plan_price"] * (1 - df["discount_percent"] / 100)
    df["is_inactive"] = (df["days_since_last_activity"] >= 14).astype(int)
    df["support_per_session"] = df["support_tickets"] / (df["sessions_last_30"] + 1)
    df["failed_payment_flag"] = (df["failed_payments"] > 0).astype(int)
    df["high_price_no_discount"] = ((df["plan_price"] >= 1590) & (df["discount_percent"] <= 5)).astype(int)
    df["low_homework_many_tickets"] = ((df["homework_completion"] < 0.45) & (df["support_tickets"] >= 3)).astype(int)
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)


## 3. Два варианта предобработки

One-Hot Encoding — понятный универсальный вариант. Target Encoding может быть компактнее, но он использует цель и поэтому должен находиться внутри pipeline. `TargetEncoder` из Scikit-Learn применяет внутреннюю кросс-валидацию при `fit_transform`.


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocess(X_part, encoding="onehot"):
    numeric_features = X_part.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_part.select_dtypes(exclude=np.number).columns.tolist()

    if encoding == "onehot":
        encoder = make_ohe()
    elif encoding == "target":
        encoder = TargetEncoder(
            target_type="binary",
            smooth="auto",
            cv=5,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Неизвестное кодирование: {encoding}")

    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", encoder),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


def make_pipeline(estimator, X_part, encoding="onehot"):
    return Pipeline(
        steps=[
            ("preprocess", make_preprocess(X_part, encoding=encoding)),
            ("model", estimator),
        ]
    )


## 4. Сравнение моделей

Все варианты оцениваются на одной стратифицированной кросс-валидации внутри train-части. Отложенная часть используется как дополнительная независимая проверка после кросс-валидации.


In [ ]:
estimators = {
    "LogisticRegression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "onehot",
    ),
    "DecisionTree": (
        DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=RANDOM_STATE),
        "onehot",
    ),
    "RandomForest": (
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "onehot",
    ),
    "ExtraTrees": (
        ExtraTreesClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "onehot",
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.04,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),
        "onehot",
    ),
    "HistGradientBoosting_TargetEncoding": (
        HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.04,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE,
        ),
        "target",
    ),
    "XGBoost": (
        XGBClassifier(
            n_estimators=220,
            learning_rate=0.04,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "onehot",
    ),
    "LightGBM": (
        LGBMClassifier(
            n_estimators=220,
            learning_rate=0.04,
            num_leaves=31,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ),
        "onehot",
    ),
    "CatBoost": (
        CatBoostClassifier(
            iterations=240,
            learning_rate=0.04,
            depth=4,
            loss_function="Logloss",
            random_seed=RANDOM_STATE,
            verbose=False,
        ),
        "onehot",
    ),
}

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
rows = []
trained = {}
val_predictions = {}

for name, (estimator, encoding) in estimators.items():
    model = make_pipeline(clone(estimator), X_train, encoding=encoding)
    cv_scores = cross_val_score(model, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=1)

    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)

    trained[name] = model
    val_predictions[name] = proba
    rows.append(
        {
            "name": name,
            "CV_ROC_AUC_mean": cv_scores.mean(),
            "CV_ROC_AUC_std": cv_scores.std(),
            "holdout_ROC_AUC": roc_auc_score(y_val, proba),
            "holdout_AP": average_precision_score(y_val, proba),
            "holdout_F1_at_0_5": f1_score(y_val, pred, zero_division=0),
        }
    )

leaderboard = pd.DataFrame(rows).sort_values("CV_ROC_AUC_mean", ascending=False)
leaderboard


## 5. Небольшой GridSearchCV

Подбираем только несколько понятных параметров случайного леса. Отложенная выборка не участвует в поиске.


In [ ]:
rf_for_grid = make_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    X_train,
    encoding="onehot",
)

param_grid = {
    "model__n_estimators": [200, 350],
    "model__max_depth": [None, 8],
    "model__min_samples_leaf": [2, 5],
}

grid = GridSearchCV(
    rf_for_grid,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=1,
    refit=True,
)
grid.fit(X_train, y_train)

grid_proba = grid.predict_proba(X_val)[:, 1]
trained["RandomForest_grid"] = grid.best_estimator_
val_predictions["RandomForest_grid"] = grid_proba

print("Лучшие параметры:", grid.best_params_)
print("Средний ROC-AUC на CV:", round(grid.best_score_, 4))
print("ROC-AUC на holdout:", round(roc_auc_score(y_val, grid_proba), 4))


## 6. Анализ ошибок и важности исходных признаков

Permutation importance считается на отложенной части и показывает падение ROC-AUC при перемешивании одного исходного столбца. Это не доказывает причинную связь, но помогает обсуждать поведение модели.


In [ ]:
analysis_model = trained["RandomForest_grid"]
analysis_proba = val_predictions["RandomForest_grid"]

errors = X_val.copy()
errors["churn_true"] = y_val
errors["churn_probability"] = analysis_proba
errors["absolute_error"] = np.abs(errors["churn_true"] - errors["churn_probability"])
display(errors.sort_values("absolute_error", ascending=False).head(10))

importance = permutation_importance(
    analysis_model,
    X_val,
    y_val,
    scoring="roc_auc",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

importance_table = (
    pd.DataFrame(
        {
            "feature": X_val.columns,
            "importance_mean": importance.importances_mean,
            "importance_std": importance.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
)
importance_table.head(12)


## 7. Проверка смеси и выбор финальной модели

Проверяем усреднение вероятностей Extra Trees, CatBoost и настроенного случайного леса. На этих данных смесь не выигрывает у лидера кросс-валидации, поэтому не используем её только потому, что она сложнее.


In [ ]:
blend_names = ["ExtraTrees", "CatBoost", "RandomForest_grid"]
avg_val_proba = np.mean([val_predictions[name] for name in blend_names], axis=0)

print("Проверяем смесь:", blend_names)
print("ROC-AUC среднего:", round(roc_auc_score(y_val, avg_val_proba), 4))
print("Average precision среднего:", round(average_precision_score(y_val, avg_val_proba), 4))

final_name = leaderboard.iloc[0]["name"]
print("Финальная модель по среднему ROC-AUC на CV:", final_name)


In [ ]:
final_model = clone(trained[final_name])
final_model.fit(X, y)
test_proba = final_model.predict_proba(test_fe[features])[:, 1]

submission = pd.DataFrame(
    {
        "id": test_fe[id_col],
        "churn_probability": test_proba.round(5),
    }
)

submission.to_csv("author_submission.csv", index=False)
submission.head()


Перед загрузкой в StackMoreLayers авторское решение проверяет число строк, уникальность `id`, отсутствие пропусков и диапазон вероятностей.


In [ ]:
assert submission.columns.tolist() == ["id", "churn_probability"]
assert len(submission) == len(test)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
assert submission["churn_probability"].notna().all()
assert submission["churn_probability"].between(0, 1).all()

print("author_submission.csv готов к загрузке в StackMoreLayers.")


## Что вынести на защиту

- бизнес-вопрос и ML-постановку;
- почему ROC-AUC подходит задаче;
- схему разбиения и защиту от утечки;
- сравнение One-Hot и Target Encoding;
- таблицу моделей и результат GridSearchCV;
- примеры крупных ошибок и важные признаки;
- финальный локальный результат и результат StackMoreLayers;
- ограничения решения и следующий эксперимент.
